In [1]:
# ============================================================
# CELL 1: IMPORTS AND CONFIG
# ============================================================
import pandas as pd
import numpy as np
import joblib
import time
import gc
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

LOCAL_DIR = Path("C:\\Users\\admin\\Documents\\Glacier Project")
REGIONS = ['R1a', 'R1b', 'R2', 'R3']
RANDOM_STATE = 42
PER_BUCKET_TARGET = 20000

EASD_FEATURES = ['elevation', 'aspect', 'slope', 'edge_distance']


In [2]:
# ============================================================
# CELL 2: LOAD DATA AND PCA ARTIFACTS
# ============================================================
t0 = time.time()
dfs = []
for region in REGIONS:
    df = pd.read_parquet(LOCAL_DIR / f'Merged\\{region.lower()}_combined.parquet')
    df['region'] = region
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(df_all):,} pixels in {time.time()-t0:.1f}s')

# Load saved PCA fit and scaler
pca = joblib.load(LOCAL_DIR / 'Saved_Models\\pca_alphaearth.joblib')
scaler = joblib.load(LOCAL_DIR / 'Saved_Models\\scaler_alphaearth.joblib')
print(f'Loaded PCA ({pca.n_components_} components) and scaler')



Loaded 15,242,639 pixels in 15.9s
Loaded PCA (64 components) and scaler


In [3]:
# ============================================================
# CELL 3: PROJECT TO PCA SPACE
# ============================================================
ae_cols = [c for c in df_all.columns 
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]
assert len(ae_cols) == 64

t0 = time.time()
X_ae = df_all[ae_cols].values.astype(np.float32)
X_scaled = scaler.transform(X_ae)
del X_ae; gc.collect()

X_pca = pca.transform(X_scaled)
del X_scaled; gc.collect()

# Attach all 10 PCs (we'll use either 5 or 10 below)
N_PCS_MAX = 10
pc_cols = [f'PC{i+1}' for i in range(N_PCS_MAX)]
df_all[pc_cols] = X_pca[:, :N_PCS_MAX]
del X_pca; gc.collect()

print(f'Projected to PC space in {time.time()-t0:.1f}s')



Projected to PC space in 23.8s


In [4]:
# ============================================================
# CELL 3b: EXPLORE edge_distance DISTRIBUTION FOR QUINTILE CUTS
# ============================================================
ed = df_all['edge_distance']

percentiles = [0, 10, 20, 25, 40, 50, 60, 75, 80, 90, 100]
print('edge_distance percentiles (full population):')
for p in percentiles:
    print(f'  p{p:3d}: {np.percentile(ed, p):8.1f}')

print(f'\nMin: {ed.min():.1f}  Max: {ed.max():.1f}  Mean: {ed.mean():.1f}  Median: {ed.median():.1f}')

print('\nedge_distance percentiles by melt_label:')
for label, grp in df_all.groupby('melt_label'):
    name = 'melt' if label == 1 else 'ice '
    vals = [np.percentile(grp['edge_distance'], p) for p in [0, 20, 40, 60, 80, 100]]
    print(f'  {name} (n={len(grp):>9,}): ' + '  '.join(f'p{p}={v:.0f}' for p, v in zip([0,20,40,60,80,100], vals)))

print('\nAuto quintile boundaries (pd.qcut on full population):')
_, auto_bins = pd.qcut(ed, q=5, retbins=True, duplicates='drop')
print('  Boundaries:', np.round(auto_bins, 1).tolist())
print('  (These split the population into 5 equal-count groups)')

print('\nPixel counts in each auto-quintile:')
q_labels = pd.qcut(ed, q=5, duplicates='drop')
print(q_labels.value_counts().sort_index())

print('\nMelt fraction per auto-quintile:')
df_all['_qtest'] = q_labels
print(df_all.groupby('_qtest')['melt_label'].mean().round(3))
df_all.drop(columns=['_qtest'], inplace=True)


edge_distance percentiles (full population):
  p  0:      9.5
  p 10:     19.5
  p 20:     37.8
  p 25:     48.1
  p 40:     87.8
  p 50:    121.3
  p 60:    164.0
  p 75:    256.9
  p 80:    301.8
  p 90:    444.4
  p100:   1691.3

Min: 9.5  Max: 1691.3  Mean: 189.4  Median: 121.3

edge_distance percentiles by melt_label:
  ice  (n=12,270,955): p0=10  p20=68  p40=125  p60=205  p80=345  p100=1691
  melt (n=2,971,684): p0=10  p20=10  p40=19  p60=28  p80=50  p100=1025

Auto quintile boundaries (pd.qcut on full population):
  Boundaries: [9.5, 37.8, 87.8, 164.0, 301.8, 1691.3]
  (These split the population into 5 equal-count groups)

Pixel counts in each auto-quintile:
edge_distance
(9.524000000000001, 37.829]    3048532
(37.829, 87.817]               3048524
(87.817, 163.99]               3048530
(163.99, 301.824]              3048526
(301.824, 1691.263]            3048527
Name: count, dtype: int64

Melt fraction per auto-quintile:


C:\Users\admin\AppData\Local\Temp\ipykernel_8896\3066642621.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_all.groupby('_qtest')['melt_label'].mean().round(3))


_qtest
(9.524000000000001, 37.829]    0.679
(37.829, 87.817]               0.224
(87.817, 163.99]               0.057
(163.99, 301.824]              0.012
(301.824, 1691.263]            0.002
Name: melt_label, dtype: float64


In [5]:
# ============================================================
# CELL 4: STRATIFIED SAMPLING (quintiles)
# ============================================================
_, quintile_bins = pd.qcut(df_all['edge_distance'], q=5, retbins=True, duplicates='drop')
quintile_bins[0] = -0.1
quintile_bins[-1] = np.inf

df_all['quintile'] = pd.cut(
    df_all['edge_distance'],
    bins=quintile_bins,
    labels=[1, 2, 3, 4, 5]
).astype(int)

print('Population per (quintile x class):')
pop = df_all.groupby(['quintile', 'melt_label']).size().unstack(fill_value=0)
pop['melt_frac'] = (pop[1] / pop.sum(axis=1)).round(3)
print(pop)
print('\nQuintile boundaries used:', np.round(quintile_bins, 1).tolist())

PER_BUCKET_TARGET = 20000

sampled = []
for (q, m), group in df_all.groupby(['quintile', 'melt_label']):
    n_take = min(PER_BUCKET_TARGET, len(group))
    sampled.append(group.sample(n=n_take, random_state=RANDOM_STATE))
df_train = pd.concat(sampled, ignore_index=True)

print(f'\nTraining set: {len(df_train):,} samples '
      f'({df_train["melt_label"].mean()*100:.1f}% melt)')
print('\nSampled per (quintile x class):')
print(df_train.groupby(['quintile', 'melt_label']).size().unstack(fill_value=0))
print('\nEdge-distance mean by class:')
print(df_train.groupby('melt_label')['edge_distance'].mean().round(1))


Population per (quintile x class):
melt_label        0        1  melt_frac
quintile                               
1            979372  2069160      0.679
2           2365708   682816      0.224
3           2873996   174534      0.057
4           3010543    37983      0.012
5           3041336     7191      0.002

Quintile boundaries used: [-0.1, 37.8, 87.8, 164.0, 301.8, inf]

Training set: 187,191 samples (46.6% melt)

Sampled per (quintile x class):
melt_label      0      1
quintile                
1           20000  20000
2           20000  20000
3           20000  20000
4           20000  20000
5           20000   7191

Edge-distance mean by class:
melt_label
0    191.100006
1    122.699997
Name: edge_distance, dtype: float32


In [8]:
# ============================================================
# CELL 4b: SINGLE RF MODEL TEST (edit FEATURES to switch)
# Uses same 80/20 split as Cell 4c so val errors are comparable.
# ============================================================
from sklearn.model_selection import train_test_split

AE_COLS       = [c for c in df_train.columns if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]
PC5           = [f'PC{i+1}' for i in range(5)]
PC10          = [f'PC{i+1}' for i in range(10)]
ELEVATION     = ['elevation']
ASPECT        = ['aspect']
SLOPE         = ['slope']
EDGE_DISTANCE = ['edge_distance']

# ── Edit this to change the model ───────────────────────────
FEATURES     = AE_COLS + ELEVATION + EDGE_DISTANCE
VAL_FRACTION = 0.2   # must match Cell 4c
# Examples:
#   Single EASD      : FEATURES = ELEVATION
#                      FEATURES = EDGE_DISTANCE
#   All EASD         : FEATURES = EASD_FEATURES
#   PCs only         : FEATURES = PC5  /  PC10
#   EASD + PCs       : FEATURES = EASD_FEATURES + PC10
#   Raw AE only      : FEATURES = AE_COLS
#   Raw AE + EASD    : FEATURES = AE_COLS + ELEVATION + EDGE_DISTANCE
#   Raw AE + all EASD: FEATURES = AE_COLS + EASD_FEATURES
# ────────────────────────────────────────────────────────────

y = df_train['melt_label'].values
X = df_train[FEATURES].values

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=VAL_FRACTION, random_state=RANDOM_STATE, stratify=y
)

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}')
print('Training...')

t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=300, oob_score=True,
    n_jobs=-1, random_state=RANDOM_STATE
)
rf.fit(X_tr, y_tr)

# OOB error (on train fold only)
oob_pred = rf.oob_decision_function_.argmax(axis=1)
cm_oob = confusion_matrix(y_tr, oob_pred)
oob_ice  = 1 - cm_oob[0, 0] / cm_oob[0].sum()
oob_melt = 1 - cm_oob[1, 1] / cm_oob[1].sum()
oob_avg  = 1 - rf.oob_score_

# Val error (held-out 20%)
val_pred = rf.predict(X_val)
cm_val = confusion_matrix(y_val, val_pred)
val_ice  = 1 - cm_val[0, 0] / cm_val[0].sum()
val_melt = 1 - cm_val[1, 1] / cm_val[1].sum()
val_avg  = (cm_val[0, 1] + cm_val[1, 0]) / cm_val.sum()

print(f'\nTrained in {time.time()-t0:.1f}s')
print(f'\n{"":12s} {"Ice err":>8} {"Melt err":>9} {"Avg err":>8}')
print(f'  OOB        {oob_ice:>8.4f} {oob_melt:>9.4f} {oob_avg:>8.4f}')
print(f'  Val        {val_ice:>8.4f} {val_melt:>9.4f} {val_avg:>8.4f}')
print(f'  Gap (V-O)  {val_ice-oob_ice:>+8.4f} {val_melt-oob_melt:>+9.4f} {val_avg-oob_avg:>+8.4f}')

print(f'\nOOB confusion matrix (rows=true, cols=pred):')
print(f'           pred-ice  pred-melt')
print(f'  true-ice  {cm_oob[0,0]:>8,}  {cm_oob[0,1]:>9,}')
print(f'  true-melt {cm_oob[1,0]:>8,}  {cm_oob[1,1]:>9,}')

print(f'\nVal confusion matrix (rows=true, cols=pred):')
print(f'           pred-ice  pred-melt')
print(f'  true-ice  {cm_val[0,0]:>8,}  {cm_val[0,1]:>9,}')
print(f'  true-melt {cm_val[1,0]:>8,}  {cm_val[1,1]:>9,}')

# Show top-20 importances (keeps output readable when using AE_COLS)
print('\nFeature importances (top 20):')
top_feats = sorted(zip(FEATURES, rf.feature_importances_), key=lambda x: -x[1])[:20]
for feat, imp in top_feats:
    bar = '█' * int(imp * 200)
    print(f'  {feat:15s} {imp:.4f}  {bar}')

Features (66): ['A00', 'A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16', 'A17', 'A18', 'A19', 'A20', 'A21', 'A22', 'A23', 'A24', 'A25', 'A26', 'A27', 'A28', 'A29', 'A30', 'A31', 'A32', 'A33', 'A34', 'A35', 'A36', 'A37', 'A38', 'A39', 'A40', 'A41', 'A42', 'A43', 'A44', 'A45', 'A46', 'A47', 'A48', 'A49', 'A50', 'A51', 'A52', 'A53', 'A54', 'A55', 'A56', 'A57', 'A58', 'A59', 'A60', 'A61', 'A62', 'A63', 'elevation', 'edge_distance']
Train: 149,752  Val: 37,439
Training...

Trained in 84.1s

              Ice err  Melt err  Avg err
  OOB          0.1439    0.1363   0.1403
  Val          0.1454    0.1364   0.1412
  Gap (V-O)   +0.0015   +0.0001  +0.0009

OOB confusion matrix (rows=true, cols=pred):
           pred-ice  pred-melt
  true-ice    68,490     11,510
  true-melt    9,507     60,245

Val confusion matrix (rows=true, cols=pred):
           pred-ice  pred-melt
  true-ice    17,092      2,908
  true-melt    2,379     15,060



In [9]:
FILE_NAME = "RF_AE64_ELE_EDGE"
joblib.dump(rf, LOCAL_DIR / f'Saved_Models\\RFs\\{FILE_NAME}.joblib')

['C:\\Users\\admin\\Documents\\Glacier Project\\Saved_Models\\RFs\\RF_AE64_ELE_EDGE.joblib']

In [48]:
# ============================================================
# CELL 4c: MLP MODEL TEST (edit FEATURES and hyperparams)
# ============================================================
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

PC5           = [f'PC{i+1}' for i in range(5)]
PC10          = [f'PC{i+1}' for i in range(10)]
ELEVATION     = ['elevation']
ASPECT        = ['aspect']
SLOPE         = ['slope']
EDGE_DISTANCE = ['edge_distance']

# ── Edit features ────────────────────────────────────────────
FEATURES = PC10 + EDGE_DISTANCE + ELEVATION
# ── Edit hyperparameters ─────────────────────────────────────
HIDDEN_LAYERS = (256, 128, 64)  # try (512,256,128) for more capacity
ACTIVATION    = 'relu'          # 'relu' | 'tanh' | 'logistic'
LEARNING_RATE = 0.001           # adam default; try 0.0003 if training is noisy
ALPHA         = 0.0001          # L2 regularisation; raise to 0.001+ if overfitting
BATCH_SIZE    = 512             # try 256 for noisier gradients, 1024 for speed
MAX_ITER      = 300             # max epochs; early stopping usually cuts this short
VAL_FRACTION  = 0.2             # held-out fraction for val error reporting
# ─────────────────────────────────────────────────────────────

y = df_train['melt_label'].values
X = df_train[FEATURES].values

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=VAL_FRACTION, random_state=RANDOM_STATE, stratify=y
)

print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Architecture : {HIDDEN_LAYERS}  activation={ACTIVATION}')
print(f'lr={LEARNING_RATE}  alpha={ALPHA}  batch={BATCH_SIZE}  max_iter={MAX_ITER}')
print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}')
print('Training...')

mlp = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS,
        activation=ACTIVATION,
        solver='adam',
        learning_rate_init=LEARNING_RATE,
        alpha=ALPHA,
        batch_size=BATCH_SIZE,
        max_iter=MAX_ITER,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=RANDOM_STATE,
        verbose=False,
    ))
])

t0 = time.time()
mlp.fit(X_tr, y_tr)
elapsed = time.time() - t0

def report(y_true, y_pred, label):
    cm = confusion_matrix(y_true, y_pred)
    ice_err  = 1 - cm[0, 0] / cm[0].sum()
    melt_err = 1 - cm[1, 1] / cm[1].sum()
    avg_err  = (cm[0, 1] + cm[1, 0]) / cm.sum()
    print(f'\n--- {label} ---')
    print(f'  Ice error  : {ice_err:.4f}  ({ice_err*100:.2f}%)')
    print(f'  Melt error : {melt_err:.4f}  ({melt_err*100:.2f}%)')
    print(f'  Avg error  : {avg_err:.4f}  ({avg_err*100:.2f}%)')
    print(f'  Confusion matrix (rows=true, cols=pred):')
    print(f'             pred-ice  pred-melt')
    print(f'    true-ice  {cm[0,0]:>8,}  {cm[0,1]:>9,}')
    print(f'    true-melt {cm[1,0]:>8,}  {cm[1,1]:>9,}')
    return avg_err

print(f'\nTrained in {elapsed:.1f}s  ({mlp.named_steps["mlp"].n_iter_} epochs)')
train_err = report(y_tr, mlp.predict(X_tr), 'TRAIN')
val_err   = report(y_val, mlp.predict(X_val), 'VAL (held-out)')
print(f'\nOverfit gap (val - train avg error): {val_err - train_err:+.4f}')


Features (12): ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'edge_distance', 'elevation']
Architecture : (256, 128, 64)  activation=relu
lr=0.001  alpha=0.0001  batch=512  max_iter=300
Train: 149,752  Val: 37,439
Training...

Trained in 177.6s  (76 epochs)

--- TRAIN ---
  Ice error  : 0.1232  (12.32%)
  Melt error : 0.1381  (13.81%)
  Avg error  : 0.1301  (13.01%)
  Confusion matrix (rows=true, cols=pred):
             pred-ice  pred-melt
    true-ice    70,143      9,857
    true-melt    9,633     60,119

--- VAL (held-out) ---
  Ice error  : 0.1737  (17.37%)
  Melt error : 0.1850  (18.50%)
  Avg error  : 0.1790  (17.90%)
  Confusion matrix (rows=true, cols=pred):
             pred-ice  pred-melt
    true-ice    16,526      3,474
    true-melt    3,226     14,213

Overfit gap (val - train avg error): +0.0488


In [49]:
FILE_NAME = "256_128_64_relu"
joblib.dump(mlp, LOCAL_DIR / f'Saved_Models\\MLPs\\{FILE_NAME}.joblib')


['C:\\Users\\admin\\Documents\\Glacier Project\\Saved_Models\\MLPs\\256_128_64_relu.joblib']

In [56]:
# ============================================================
# CELL 4d: RF vs MLP — fair comparison on same val split
# FEATURES must match exactly what was used in Cell 4c (same
# order), otherwise the MLP scaler will apply wrong transforms.
# ============================================================
from sklearn.model_selection import train_test_split

FEATURES     = PC10 + EDGE_DISTANCE + ELEVATION  # must match Cell 4c exactly
VAL_FRACTION = 0.2                                # must match Cell 4c

y = df_train['melt_label'].values
X = df_train[FEATURES].values

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=VAL_FRACTION, random_state=RANDOM_STATE, stratify=y
)
print(f'Features ({len(FEATURES)}): {FEATURES}')
print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}\n')

# Load MLP and verify it is actually an MLP pipeline
mlp_cmp = joblib.load(LOCAL_DIR / 'Saved_Models\\MLPs\\256_128_64_relu.joblib')
print(f'Loaded model type: {type(mlp_cmp)}')

def val_errors(name, model):
    cm = confusion_matrix(y_val, model.predict(X_val))
    ice_err  = 1 - cm[0, 0] / cm[0].sum()
    melt_err = 1 - cm[1, 1] / cm[1].sum()
    avg_err  = (cm[0, 1] + cm[1, 0]) / cm.sum()
    print(f'  {name:8s}  ice={ice_err:.4f}  melt={melt_err:.4f}  avg={avg_err:.4f}')

# Train RF on X_tr only so val set is genuinely held-out
print('Training RF on X_tr...')
t0 = time.time()
rf_cmp = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE)
rf_cmp.fit(X_tr, y_tr)
print(f'  Done in {time.time()-t0:.1f}s\n')

print(f'Val set: {len(X_val):,} samples')
print(f'{"":8s}  {"ICE err":>8}  {"MELT err":>9}  {"AVG err":>8}')
val_errors('RF',  rf_cmp)
val_errors('MLP', mlp_cmp)


Features (12): ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10', 'edge_distance', 'elevation']
Train: 149,752  Val: 37,439

Loaded model type: <class 'sklearn.pipeline.Pipeline'>
Training RF on X_tr...
  Done in 82.9s

Val set: 37,439 samples
           ICE err   MELT err   AVG err
  RF        ice=0.1597  melt=0.1547  avg=0.1573
  MLP       ice=0.1737  melt=0.1850  avg=0.1790


In [60]:
# ============================================================
# CELL 4e: MLP on raw AlphaEarth embeddings (all 64 dims, no PCA)
# ============================================================
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# Derive AE column names the same way Cell 3 does
AE_COLS       = [c for c in df_train.columns if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]
ELEVATION     = ['elevation']
ASPECT        = ['aspect']
SLOPE         = ['slope']
EDGE_DISTANCE = ['edge_distance']
print(f'Found {len(AE_COLS)} AE columns: {AE_COLS[0]} … {AE_COLS[-1]}')

# ── Edit features ────────────────────────────────────────────
FEATURES = AE_COLS + EDGE_DISTANCE + ELEVATION
# Examples:
#   AE only              : FEATURES = AE_COLS
#   AE + edge distance   : FEATURES = AE_COLS + EDGE_DISTANCE
#   AE + elevation       : FEATURES = AE_COLS + ELEVATION
#   AE + all EASD        : FEATURES = AE_COLS + EASD_FEATURES
# ── Edit hyperparameters ─────────────────────────────────────
HIDDEN_LAYERS = (256, 128, 64)  # try (512, 256, 128) for more capacity with 64 inputs
ACTIVATION    = 'relu'
LEARNING_RATE = 0.001
ALPHA         = 0.0001
BATCH_SIZE    = 512
MAX_ITER      = 300
VAL_FRACTION  = 0.2
# ─────────────────────────────────────────────────────────────

y = df_train['melt_label'].values
X = df_train[FEATURES].values.astype(np.float32)

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=VAL_FRACTION, random_state=RANDOM_STATE, stratify=y
)

print(f'Features ({len(FEATURES)}): {FEATURES[:3]} … {FEATURES[-3:]}')
print(f'Architecture : {HIDDEN_LAYERS}  activation={ACTIVATION}')
print(f'lr={LEARNING_RATE}  alpha={ALPHA}  batch={BATCH_SIZE}  max_iter={MAX_ITER}')
print(f'Train: {len(X_tr):,}  Val: {len(X_val):,}')
print('Training...')

mlp_ae = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYERS,
        activation=ACTIVATION,
        solver='adam',
        learning_rate_init=LEARNING_RATE,
        alpha=ALPHA,
        batch_size=BATCH_SIZE,
        max_iter=MAX_ITER,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        random_state=RANDOM_STATE,
        verbose=False,
    ))
])

t0 = time.time()
mlp_ae.fit(X_tr, y_tr)
elapsed = time.time() - t0

def report_ae(y_true, y_pred, label):
    cm = confusion_matrix(y_true, y_pred)
    ice_err  = 1 - cm[0, 0] / cm[0].sum()
    melt_err = 1 - cm[1, 1] / cm[1].sum()
    avg_err  = (cm[0, 1] + cm[1, 0]) / cm.sum()
    print(f'\n--- {label} ---')
    print(f'  Ice error  : {ice_err:.4f}  ({ice_err*100:.2f}%)')
    print(f'  Melt error : {melt_err:.4f}  ({melt_err*100:.2f}%)')
    print(f'  Avg error  : {avg_err:.4f}  ({avg_err*100:.2f}%)')
    print(f'  Confusion matrix (rows=true, cols=pred):')
    print(f'             pred-ice  pred-melt')
    print(f'    true-ice  {cm[0,0]:>8,}  {cm[0,1]:>9,}')
    print(f'    true-melt {cm[1,0]:>8,}  {cm[1,1]:>9,}')
    return avg_err

print(f'\nTrained in {elapsed:.1f}s  ({mlp_ae.named_steps["mlp"].n_iter_} epochs)')
train_err = report_ae(y_tr,  mlp_ae.predict(X_tr),  'TRAIN')
val_err   = report_ae(y_val, mlp_ae.predict(X_val), 'VAL (held-out)')
print(f'\nOverfit gap (val - train avg error): {val_err - train_err:+.4f}')


Found 64 AE columns: A00 … A63
Features (66): ['A00', 'A01', 'A02'] … ['A63', 'edge_distance', 'elevation']
Architecture : (256, 128, 64)  activation=relu
lr=0.001  alpha=0.0001  batch=512  max_iter=300
Train: 149,752  Val: 37,439
Training...

Trained in 295.6s  (46 epochs)

--- TRAIN ---
  Ice error  : 0.0848  (8.48%)
  Melt error : 0.0872  (8.72%)
  Avg error  : 0.0859  (8.59%)
  Confusion matrix (rows=true, cols=pred):
             pred-ice  pred-melt
    true-ice    73,212      6,788
    true-melt    6,081     63,671

--- VAL (held-out) ---
  Ice error  : 0.1382  (13.82%)
  Melt error : 0.1400  (14.00%)
  Avg error  : 0.1391  (13.91%)
  Confusion matrix (rows=true, cols=pred):
             pred-ice  pred-melt
    true-ice    17,236      2,764
    true-melt    2,442     14,997

Overfit gap (val - train avg error): +0.0531


In [61]:
FILE_NAME = "256_128_64_relu_64dim_ee"
joblib.dump(mlp, LOCAL_DIR / f'Saved_Models\\MLPs\\{FILE_NAME}.joblib')


['C:\\Users\\admin\\Documents\\Glacier Project\\Saved_Models\\MLPs\\256_128_64_relu_64dim_ee.joblib']

In [58]:
#Load MLP from disk
mlp_cmp = joblib.load(LOCAL_DIR / 'Saved_Models\\MLPs\\256_128_64_relu.joblib')

print(f'\nVal set: {len(X_val):,} samples')
print(f'{"":8s}  {"ICE err":>8}  {"MELT err":>9}  {"AVG err":>8}')
val_errors('MLP', mlp_cmp)
val_errors('RF',  rf_cmp)


Val set: 37,439 samples
           ICE err   MELT err   AVG err


ValueError: X has 64 features, but StandardScaler is expecting 12 features as input.

In [28]:
# ============================================================
# CELL 5: TRAIN AND EVALUATE THREE MODELS
# ============================================================
y = df_train['melt_label'].values

def train_and_eval(X, y, label):
    t0 = time.time()
    rf = RandomForestClassifier(
        n_estimators=300, oob_score=True,
        n_jobs=-1, random_state=RANDOM_STATE
    )
    rf.fit(X, y)
    oob_pred = rf.oob_decision_function_.argmax(axis=1)
    cm = confusion_matrix(y, oob_pred)
    ice_err = 1 - cm[0, 0] / cm[0].sum()
    melt_err = 1 - cm[1, 1] / cm[1].sum()
    avg_err = 1 - rf.oob_score_
    
    print(f'\n=== {label} (trained in {time.time()-t0:.1f}s) ===')
    print(f'  Ice error  : {ice_err:.4f}')
    print(f'  Melt error : {melt_err:.4f}')
    print(f'  Avg error  : {avg_err:.4f}')
    return rf, ice_err, melt_err, avg_err

# Model 1: EASD-only (new baseline with own labels)
X_easd = df_train[EASD_FEATURES].values
rf_easd, ie1, me1, ae1 = train_and_eval(X_easd, y, 'EASD-only (new baseline)')

# Model 2: EASD + PC1-PC5
PC5 = [f'PC{i+1}' for i in range(5)]
X_pc5 = df_train[EASD_FEATURES + PC5].values
rf_pc5, ie2, me2, ae2 = train_and_eval(X_pc5, y, 'EASD + PC1-PC5')

# Model 3: EASD + PC1-PC10
PC10 = [f'PC{i+1}' for i in range(10)]
X_pc10 = df_train[EASD_FEATURES + PC10].values
rf_pc10, ie3, me3, ae3 = train_and_eval(X_pc10, y, 'EASD + PC1-PC10')

# Summary table
print('\n=== SUMMARY ===')
print(f'{"Model":<25}{"Ice":<10}{"Melt":<10}{"Avg":<10}{"ΔAvg vs baseline":<10}')
print(f'{"EASD-only":<25}{ie1:<10.4f}{me1:<10.4f}{ae1:<10.4f}{0:<10.4f}')
print(f'{"EASD + PC1-PC5":<25}{ie2:<10.4f}{me2:<10.4f}{ae2:<10.4f}{ae2-ae1:<+10.4f}')
print(f'{"EASD + PC1-PC10":<25}{ie3:<10.4f}{me3:<10.4f}{ae3:<10.4f}{ae3-ae1:<+10.4f}')

# Feature importances
print('\n=== Feature importances (PC10 model) ===')
all_features = EASD_FEATURES + PC10
for feat, imp in sorted(zip(all_features, rf_pc10.feature_importances_),
                        key=lambda x: -x[1]):
    print(f'  {feat:15s} {imp:.4f}')

KeyboardInterrupt: 

In [4]:
import joblib
import numpy as np
import pandas as pd
import time
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix

# ============================================================
# CONFIG
# ============================================================
LOCAL_DIR = Path("C:\\Users\\admin\\Documents\\Glacier Project")
REGIONS = ['R1a', 'R1b', 'R2', 'R3']
RANDOM_STATE = 42

EASD_FEATURES_TO_USE = ['elevation', 'edge_distance']  # NOT aspect or slope
# AE columns are added below dynamically

# ============================================================
# LOAD DATA AND STRATIFY (same as your locked pipeline)
# ============================================================
print('Loading data...')
dfs = []
for region in REGIONS:
    df = pd.read_parquet(LOCAL_DIR / f'Merged\\{region.lower()}_combined.parquet')
    df['region'] = region
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)

ae_cols = [c for c in df_all.columns 
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]
print(f'Found {len(ae_cols)} AE bands')

# 5-quintile stratification with 20k per bucket (capped at availability)
df_all['quintile'] = pd.qcut(df_all['edge_distance'], q=5, labels=False) + 1

PER_BUCKET_TARGET = 20000
sampled = []
for q, q_group in df_all.groupby('quintile'):
    n_melt = (q_group['melt_label'] == 1).sum()
    n_nonmelt = (q_group['melt_label'] == 0).sum()
    n_take_melt = min(PER_BUCKET_TARGET, n_melt)
    n_take_nonmelt = min(PER_BUCKET_TARGET, n_nonmelt)
    
    melt_sample = q_group[q_group['melt_label'] == 1].sample(
        n=n_take_melt, random_state=RANDOM_STATE)
    nonmelt_sample = q_group[q_group['melt_label'] == 0].sample(
        n=n_take_nonmelt, random_state=RANDOM_STATE)
    sampled.append(pd.concat([melt_sample, nonmelt_sample]))

df_train_full = pd.concat(sampled, ignore_index=True)
print(f'Stratified training set: {len(df_train_full):,} samples '
      f'({df_train_full["melt_label"].mean()*100:.1f}% melt)')

# Define the 66 features in a stable order
FEATURE_COLS = EASD_FEATURES_TO_USE + ae_cols
assert len(FEATURE_COLS) == 66, f'Expected 66 features, got {len(FEATURE_COLS)}'

# ============================================================
# TRAIN/VAL SPLIT
# ============================================================
X = df_train_full[FEATURE_COLS].values
y = df_train_full['melt_label'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {len(X_train):,}, Val: {len(X_val):,}')

# ============================================================
# STANDARDISE - FIT ON TRAIN ONLY
# Save the fitted scaler so we can apply it to Peru-wide data later
# ============================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Save the scaler immediately (before training, in case training crashes)
joblib.dump(scaler, LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_scaler.joblib')
print('Saved scaler to mlp_scaler.joblib')

# ============================================================
# TRAIN THE BEST MLP CONFIGURATION
# ============================================================
print('\nTraining MLP...')
t0 = time.time()

mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    solver='adam',
    learning_rate_init=0.001,
    alpha=0.0001,
    batch_size=512,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=10,
    random_state=RANDOM_STATE,
    verbose=False
)
mlp.fit(X_train_scaled, y_train)

print(f'Trained in {time.time()-t0:.1f}s ({mlp.n_iter_} epochs)')

# ============================================================
# EVALUATE ON VAL SET (sanity check matches your earlier result)
# ============================================================
y_pred = mlp.predict(X_val_scaled)
cm = confusion_matrix(y_val, y_pred)
ice_err = 1 - cm[0,0]/cm[0].sum()
melt_err = 1 - cm[1,1]/cm[1].sum()
avg_err = 1 - (cm[0,0] + cm[1,1])/cm.sum()

print(f'\nValidation set performance:')
print(f'  Ice error:  {ice_err:.4f}')
print(f'  Melt error: {melt_err:.4f}')
print(f'  Avg error:  {avg_err:.4f}')
print(f'\n(Expected from your earlier seed-42 run: ~0.1346)')

# ============================================================
# SAVE EVERYTHING
# ============================================================
joblib.dump(mlp, LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_best.joblib')
print(f'\nSaved MLP to mlp_best.joblib')

# Save feature column list - critical for applying to Peru-wide data
# This guarantees the same column order is used during prediction
with open(LOCAL_DIR / 'Saved_Models\\MLPs\\mlp_feature_cols.txt', 'w') as f:
    for col in FEATURE_COLS:
        f.write(f'{col}\n')
print(f'Saved feature column list to mlp_feature_cols.txt')

print('\nAll artifacts saved. Ready for spatial overlap pipeline.')

Loading data...
Found 64 AE bands
Stratified training set: 187,191 samples (46.6% melt)
Train: 149,752, Val: 37,439
Saved scaler to mlp_scaler.joblib

Training MLP...
Trained in 596.0s (35 epochs)

Validation set performance:
  Ice error:  0.1321
  Melt error: 0.1399
  Avg error:  0.1358

(Expected from your earlier seed-42 run: ~0.1346)

Saved MLP to mlp_best.joblib
Saved feature column list to mlp_feature_cols.txt

All artifacts saved. Ready for spatial overlap pipeline.
